In [1]:
# CSV 데이터를 표 형태로 처리하기 위해 pandas를 가져옵니다.
import pandas as pd

# Hugging Face 비공개 데이터셋에서 파일을 받는 함수를 가져옵니다.
from huggingface_hub import hf_hub_download

# 프로젝트 루트의 .env에서 인증 토큰을 읽는 공통 모듈을 가져옵니다.
from common import secrets

# 프로젝트 루트의 .env에서 본인의 Hugging Face 토큰을 읽습니다.
token, token_source = secrets.load_key(
    ("HUGGINGFACE_ACCESS_TOKEN",),
)

# 토큰이 비어 있다면 잘못된 인증 요청을 보내지 않고 즉시 알려줍니다.
if not token:
    raise RuntimeError(
        "HUGGINGFACE_ACCESS_TOKEN을 찾지 못했습니다. "
        "프로젝트 루트의 .env 파일을 확인해주세요."
    )

# Hugging Face 토큰은 hf_로 시작하는 ASCII 문자열이어야 합니다.
#
# 이 검사를 통해 .env의 한글 주석이나 예시 문장을
# 토큰으로 잘못 읽는 문제를 다운로드 전에 발견합니다.
if not token.startswith("hf_") or not token.isascii():
    raise RuntimeError(
        "Hugging Face 토큰 형식이 올바르지 않습니다. "
        ".env의 HUGGINGFACE_ACCESS_TOKEN 줄에는 실제 토큰만 입력해주세요."
    )

# 팀의 비공개 Hugging Face 데이터셋에서
# 준영님에게 제공된 KOSPI200 피처·라벨 CSV 파일을 받습니다.
#
# 파일은 프로젝트 폴더가 아니라 Hugging Face 캐시에 저장되므로
# CSV 원본이 실수로 GitHub에 커밋되지 않습니다.
data_path = hf_hub_download(
    repo_id="qurious-quant/alphastack-krx-dev",
    filename="small/features_labels_kospi200_dev.csv",
    repo_type="dataset",
    token=token,
)

# 기준일이 숫자로 변형되지 않도록 bas_dd를 문자열로 지정해 CSV를 읽습니다.
df = pd.read_csv(
    data_path,
    dtype={"bas_dd": "string"},
)

# 토큰 전체는 출력하지 않고 어느 파일에서 읽었는지만 확인합니다.
print("토큰 출처:", token_source)

# 데이터의 행 개수와 열 개수를 확인합니다.
print("전체 데이터 크기:", df.shape)

# 모델 입력 X에 넣지 않을 열을 지정합니다.
#
# 날짜·지수 이름은 모델이 학습할 수치 피처가 아니므로 제외합니다.
# 시가·고가·저가·종가 등의 원본값도 현재 기준 모델 피처에서 제외합니다.
# fwd_return_5d와 label은 미래 정보를 담고 있으므로 반드시 X에서 제외합니다.
NOT_FEATURE = {
    "bas_dd",
    "date",
    "index_name",
    "index_class",
    "open",
    "high",
    "low",
    "close",
    "change",
    "change_rate",
    "volume",
    "value",
    "market_cap",
    "fwd_return_5d",
    "label",
}

# 모델 학습에 반드시 필요한 기준일과 정답 열이 존재하는지 확인합니다.
REQUIRED_COLUMNS = {
    "bas_dd",
    "label",
}

# 필수 열 중 데이터에 없는 열을 찾습니다.
missing_columns = REQUIRED_COLUMNS - set(df.columns)

# 필수 열이 없다면 이후 학습을 진행하지 않고 누락된 열을 알려줍니다.
if missing_columns:
    raise ValueError(
        f"필수 열이 없습니다: {sorted(missing_columns)}"
    )

# 제외 대상이 아닌 나머지 열을 모델 입력 피처로 선택합니다.
#
# CSV에 저장된 기존 열 순서를 그대로 유지합니다.
FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in NOT_FEATURE
]

# 선택한 피처들로 모델 입력값 X를 만듭니다.
X = df.loc[:, FEATURE_COLUMNS].copy()

# 미래 5거래일 방향 라벨을 모델의 정답 y로 만듭니다.
#
# 하락=-1, 중립=0, 상승=1입니다.
y = df["label"].copy()

# 선택된 피처 개수와 이름을 확인합니다.
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("피처 개수:", len(FEATURE_COLUMNS))
print("피처 목록:", FEATURE_COLUMNS)

# X에 학습할 행이나 피처가 하나도 없는지 확인합니다.
if X.empty:
    raise ValueError("모델에 사용할 X 데이터가 비어 있습니다.")

# 원본 라벨에 결측값이 있는지 확인합니다.
missing_label_count = int(y.isna().sum())

if missing_label_count > 0:
    raise ValueError(
        f"y에 결측 라벨이 {missing_label_count}개 있습니다."
    )

# CSV의 한글 라벨 앞뒤에 불필요한 공백이 있을 가능성을 제거합니다.
y_text = y.astype("string").str.strip()

# Hugging Face 데이터셋에서 사용하는 한글 라벨을 지정합니다.
EXPECTED_TEXT_LABELS = {
    "하락",
    "중립",
    "상승",
}

# 실제 데이터에 들어 있는 라벨을 확인합니다.
observed_text_labels = set(
    y_text.dropna().unique().tolist()
)

# 정해진 세 라벨에 포함되지 않는 값이 있는지 확인합니다.
unexpected_labels = observed_text_labels - EXPECTED_TEXT_LABELS

if unexpected_labels:
    raise ValueError(
        "알 수 없는 라벨이 포함되어 있습니다: "
        f"{sorted(unexpected_labels)}"
    )

# CSV의 한글 라벨을 모델이 사용할 숫자 라벨로 변환합니다.
#
# 하락=-1
# 중립=0
# 상승=1
LABEL_TO_NUMBER = {
    "하락": -1,
    "중립": 0,
    "상승": 1,
}

# 한글 라벨을 숫자 라벨로 변환합니다.
y = y_text.map(LABEL_TO_NUMBER)

# 변환되지 않은 라벨이 남아 있는지 확인합니다.
#
# 한글 오타나 예상하지 못한 값이 있다면 map() 결과가 결측값이 됩니다.
unmapped_label_count = int(y.isna().sum())

if unmapped_label_count > 0:
    raise ValueError(
        f"숫자로 변환하지 못한 라벨이 {unmapped_label_count}개 있습니다."
    )

# 모든 라벨이 정상적으로 변환된 뒤 정수형으로 변경합니다.
y = y.astype(int)

# 변환된 숫자 라벨의 종류를 확인합니다.
observed_numeric_labels = sorted(y.unique().tolist())

# 프로젝트가 사용하는 세 클래스와 일치하는지 확인합니다.
if observed_numeric_labels != [-1, 0, 1]:
    raise ValueError(
        "숫자 라벨은 하락=-1, 중립=0, 상승=1이어야 합니다. "
        f"현재 라벨: {observed_numeric_labels}"
    )

# X에 문자열처럼 모델이 바로 학습할 수 없는 열이 있는지 확인합니다.
non_numeric_features = [
    column
    for column in FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(X[column])
]

if non_numeric_features:
    raise TypeError(
        "숫자가 아닌 피처가 포함되어 있습니다: "
        f"{non_numeric_features}"
    )

# 전체 피처에 포함된 결측값 개수를 계산합니다.
missing_feature_count = int(X.isna().sum().sum())

# 라벨 분포는 사람이 읽기 쉬운 한글 라벨을 기준으로 계산합니다.
#
# reindex를 사용해 항상 하락·중립·상승 순서로 출력되도록 합니다.
label_distribution = (
    y_text.value_counts(normalize=True)
    .reindex(["하락", "중립", "상승"], fill_value=0)
    .mul(100)
    .round(2)
)

# 원본 행은 출력하지 않고 학습에 필요한 요약 정보만 확인합니다.
print("데이터 시작일:", df["bas_dd"].min())
print("데이터 종료일:", df["bas_dd"].max())
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("X 전체 결측값:", missing_feature_count)
print("숫자 라벨:", observed_numeric_labels)
print("라벨 변환:", LABEL_TO_NUMBER)
print("라벨 분포(%):")
print(label_distribution)

C:\Users\Administrator\Alpha_Stack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


토큰 출처: .env
전체 데이터 크기: (2815, 37)
X 크기: (2815, 22)
y 크기: (2815,)
피처 개수: 22
피처 목록: ['sma_5', 'sma_20', 'sma_60', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_bandwidth', 'true_range', 'atr_14', 'hv_20', 'parkinson_20', 'vol_sma_20', 'vol_ratio_20', 'obv', 'vwap_20', 'vol_roc_5']
데이터 시작일: 20100330
데이터 종료일: 20210823
X 크기: (2815, 22)
y 크기: (2815,)
X 전체 결측값: 0
숫자 라벨: [-1, 0, 1]
라벨 변환: {'하락': -1, '중립': 0, '상승': 1}
라벨 분포(%):
label
하락    27.32
중립    38.69
상승     34.0
Name: proportion, dtype: double[pyarrow]


# XGBoost — 조합A + Daily_Return

## 실험 목적

기존 조합 A에 `daily_return` 수익률 피처를 추가해 같은 12개
워크포워드 폴드에서 성능 변화를 확인한다. 수익률은 현재와 과거 종가만 사용하며,
모델의 클래스 가중치는 기존 조합에서 선택한 **balanced** 설정을 유지한다.


In [2]:
# 프로젝트 루트를 sys.path에 넣은 뒤 내부 모듈을 가져와야 합니다.
# ruff: noqa: E402

# 프로젝트 모듈을 불러오기 위해 현재 위치에서 프로젝트 루트를 찾습니다.
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

while (
    project_root != project_root.parent
    and not (project_root / "models").is_dir()
):
    project_root = project_root.parent

if not (project_root / "models").is_dir():
    raise RuntimeError("프로젝트 루트의 models 폴더를 찾지 못했습니다.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 배열과 결과표 계산에 필요한 라이브러리를 가져옵니다.
import numpy as np
import pandas as pd

# 3개 클래스 분류 모델의 성능을 계산할 함수를 가져옵니다.
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# 프로젝트에서 작성한 기준선 예측 함수를 가져옵니다.
from evaluation.baseline import (
    always_up,
    majority_class,
)

# 프로젝트에서 작성한 방향 적중률과 워크포워드 분할 함수를 가져옵니다.
from evaluation.metrics import hit_rate
from evaluation.walk_forward import expanding_splits

# 앞에서 작성한 XGBoost 모델 생성 함수를 가져옵니다.
from models.xgboost import build_xgboost_baseline

print("프로젝트 루트:", project_root)

프로젝트 루트: C:\Users\Administrator\Alpha_Stack


In [3]:
# 조합 A에서 사용할 네 가지 피처를 지정합니다.
COMBINATION_A = [
    "rsi_14",
    "bb_bandwidth",
    "hv_20",
    "vol_ratio_20",
]

# 실험에 필요한 설정값을 지정합니다.
N_FOLDS = 12
MIN_TRAIN_SIZE = 750
VALID_SIZE = 60
LABEL_HORIZON = 5

# 결과표에서 사용할 클래스 순서와 이름을 지정합니다.
CLASS_LABELS = [-1, 0, 1]

CLASS_NAMES = {
    -1: "하락",
    0: "중립",
    1: "상승",
}

# 위쪽 데이터 준비 과정에서 만든 전체 X 중
# 조합 A의 네 가지 피처만 선택합니다.
X_combination_a = X.loc[:, COMBINATION_A].astype(float).copy()

# 현재와 과거 종가만 사용하는 수익률 피처를 추가합니다.
return_close = pd.to_numeric(df["close"], errors="raise").astype(float)
X_combination_a["daily_return"] = return_close / return_close.shift(1) - 1.0
COMBINATION_A = [*COMBINATION_A, "daily_return"]

# 한글 라벨에서 숫자로 변환된 y를 정수 배열로 준비합니다.
y_numeric = y.astype(int).to_numpy()

# 데이터가 날짜 오름차순으로 정렬되어 있는지 확인합니다.
if not df["bas_dd"].is_monotonic_increasing:
    raise RuntimeError("데이터가 날짜 오름차순으로 정렬되어 있지 않습니다.")

# 조합 A에 필요한 피처가 모두 존재하는지 확인합니다.
missing_features = [
    feature
    for feature in COMBINATION_A
    if feature not in X_combination_a.columns
]

if missing_features:
    raise RuntimeError(
        f"데이터에 없는 피처가 있습니다: {missing_features}"
    )

# 조합 A에 결측값이 있는지 확인합니다.
missing_value_count = int(
    X_combination_a.isna().sum().sum()
)

expected_warmup_missing = 1
if missing_value_count != expected_warmup_missing:
    raise RuntimeError(
        f"수익률 워밍업 외 결측값이 있습니다: {missing_value_count}개"
    )

# X의 행 개수와 y의 개수가 같은지 확인합니다.
if len(X_combination_a) != len(y_numeric):
    raise RuntimeError(
        "X와 y의 데이터 개수가 서로 다릅니다."
    )

print("실험 모델: XGBoost")
print("피처 조합: A")
print("사용 피처:", COMBINATION_A)
print("X 크기:", X_combination_a.shape)
print("y 크기:", y_numeric.shape)
print("X 전체 결측값:", missing_value_count)
print("숫자 라벨:", sorted(np.unique(y_numeric).tolist()))

실험 모델: XGBoost
피처 조합: A
사용 피처: ['rsi_14', 'bb_bandwidth', 'hv_20', 'vol_ratio_20', 'daily_return']
X 크기: (2815, 5)
y 크기: (2815,)
X 전체 결측값: 1
숫자 라벨: [-1, 0, 1]


In [4]:
# 확장형 워크포워드 학습·검증 구간을 만듭니다.
#
# horizon=60은 폴드마다 검증할 데이터의 개수입니다.
# label_horizon=5는 y가 미래 5거래일을 사용한다는 의미입니다.
# gap=5로 설정하여 학습 라벨과 검증 구간이 겹치지 않게 합니다.
splits = expanding_splits(
    n_samples=len(X_combination_a),
    n_folds=N_FOLDS,
    min_train=MIN_TRAIN_SIZE,
    horizon=VALID_SIZE,
    gap=LABEL_HORIZON,
    label_horizon=LABEL_HORIZON,
)

# 요청한 12개 폴드가 생성되었는지 확인합니다.
if len(splits) != N_FOLDS:
    raise RuntimeError(
        f"워크포워드 폴드가 {N_FOLDS}개가 아니라 "
        f"{len(splits)}개 생성되었습니다."
    )

# 각 폴드가 시간순으로 안전하게 분리되었는지 확인합니다.
for fold_number, (train_index, valid_index) in enumerate(
    splits,
    start=1,
):
    # 학습 마지막 행과 검증 첫 행 사이에서 제외된 행 개수를 계산합니다.
    actual_gap = (
        int(valid_index[0])
        - int(train_index[-1])
        - 1
    )

    if train_index[-1] >= valid_index[0]:
        raise RuntimeError(
            f"{fold_number}번 폴드에서 "
            "학습과 검증의 시간 순서가 잘못되었습니다."
        )

    if actual_gap != LABEL_HORIZON:
        raise RuntimeError(
            f"{fold_number}번 폴드의 gap이 "
            f"{actual_gap}개입니다."
        )

print("워크포워드 폴드 수:", len(splits))
print("최초 학습 데이터 수:", len(splits[0][0]))
print("폴드별 검증 데이터 수:", len(splits[0][1]))
print("학습·검증 사이 gap:", LABEL_HORIZON)

워크포워드 폴드 수: 12
최초 학습 데이터 수: 750
폴드별 검증 데이터 수: 60
학습·검증 사이 gap: 5


In [5]:
# 폴드별 평가 결과를 저장할 목록입니다.
fold_results = []

# 모든 검증 구간의 실제값과 예측값을 저장할 목록입니다.
all_y_true = []
all_y_pred = []

# 기준선의 예측값도 같은 검증 구간 순서로 저장합니다.
all_always_up_pred = []
all_direction_majority_pred = []
all_three_class_majority_pred = []

# 12개의 워크포워드 폴드를 시간순으로 학습합니다.
for fold_number, (train_index, valid_index) in enumerate(
    splits,
    start=1,
):
    # 현재 폴드의 학습 X와 검증 X를 선택합니다.
    X_train = X_combination_a.iloc[train_index]
    X_valid = X_combination_a.iloc[valid_index]

    # 현재 폴드의 학습 y와 검증 y를 선택합니다.
    y_train = y_numeric[train_index]
    y_valid = y_numeric[valid_index]

    # 폴드마다 새로운 XGBoost 모델을 생성합니다.
    #
    # 트리 모델은 StandardScaler 없이 현재 폴드의 학습 데이터만 사용합니다.
    # 검증 데이터의 분포나 정답은 학습 과정에 전달하지 않습니다.
    # 폴드별 학습 라벨 빈도의 역수를 표본 가중치로 변환해 사용합니다.
    model = build_xgboost_baseline(class_weight="balanced")

    # 현재 폴드의 과거 학습 데이터만 사용해 모델을 학습합니다.
    # 수익률 워밍업 결측은 폴드 안에서만 제외해 검증 기간을 유지합니다.
    train_usable = np.isfinite(X_train.to_numpy(dtype=float)).all(axis=1)
    valid_usable = np.isfinite(X_valid.to_numpy(dtype=float)).all(axis=1)
    X_train = X_train.loc[train_usable]
    y_train = y_train[train_usable]
    X_valid = X_valid.loc[valid_usable]
    y_valid = y_valid[valid_usable]

    model.fit(X_train, y_train)

    # 학습하지 않은 미래 검증 구간을 예측합니다.
    y_pred = model.predict(X_valid)

    # 검증 행마다 하락·중립·상승 확률을 계산합니다.
    probabilities = model.predict_proba(X_valid)

    # 각 행의 클래스 확률 합이 1인지 확인합니다.
    if not np.allclose(
        probabilities.sum(axis=1),
        np.ones(len(X_valid)),
    ):
        raise RuntimeError(
            f"{fold_number}번 폴드의 예측 확률 합이 1이 아닙니다."
        )

    # 항상 상승이라고 예측하는 기준선을 만듭니다.
    always_up_pred = always_up(len(y_valid))

    # 학습 구간에서 상승과 하락 중 더 많았던 방향을
    # 검증 구간 전체에 예측하는 기준선을 만듭니다.
    direction_majority_pred = majority_class(
        y_train,
        len(y_valid),
    )

    # 중립까지 포함한 3개 클래스 중 학습 구간에서
    # 가장 많았던 클래스를 찾습니다.
    train_labels, train_counts = np.unique(
        y_train,
        return_counts=True,
    )

    three_class_majority_label = int(
        train_labels[np.argmax(train_counts)]
    )

    # 학습 구간에서 가장 많았던 3분류 클래스로만 예측합니다.
    three_class_majority_pred = np.full(
        shape=len(y_valid),
        fill_value=three_class_majority_label,
        dtype=int,
    )

    # 현재 폴드의 성능을 계산해 저장합니다.
    fold_results.append(
        {
            "fold": fold_number,
            "train_size": len(train_index),
            "valid_size": len(valid_index),
            "train_end": df.iloc[train_index[-1]]["bas_dd"],
            "valid_start": df.iloc[valid_index[0]]["bas_dd"],
            "valid_end": df.iloc[valid_index[-1]]["bas_dd"],
            "accuracy": accuracy_score(
                y_valid,
                y_pred,
            ),
            "macro_f1": f1_score(
                y_valid,
                y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_valid,
                y_pred,
            ),
            "direction_hit_rate": hit_rate(
                y_pred,
                y_valid,
            ),
            "always_up_accuracy": accuracy_score(
                y_valid,
                always_up_pred,
            ),
            "three_class_majority_accuracy": accuracy_score(
                y_valid,
                three_class_majority_pred,
            ),
            "always_up_direction_hit_rate": hit_rate(
                always_up_pred,
                y_valid,
            ),
            "direction_majority_hit_rate": hit_rate(
                direction_majority_pred,
                y_valid,
            ),
        }
    )

    # 전체 OOS 성능을 계산하기 위해 현재 폴드 결과를 저장합니다.
    all_y_true.append(y_valid)
    all_y_pred.append(y_pred)
    all_always_up_pred.append(always_up_pred)
    all_direction_majority_pred.append(
        direction_majority_pred
    )
    all_three_class_majority_pred.append(
        three_class_majority_pred
    )

    print(
        f"{fold_number:02d}번 폴드 완료 | "
        f"학습 {len(train_index)}개 | "
        f"검증 {len(valid_index)}개"
    )

print("전체 워크포워드 학습이 완료되었습니다.")

01번 폴드 완료 | 학습 750개 | 검증 60개


02번 폴드 완료 | 학습 932개 | 검증 60개


03번 폴드 완료 | 학습 1114개 | 검증 60개


04번 폴드 완료 | 학습 1295개 | 검증 60개


05번 폴드 완료 | 학습 1477개 | 검증 60개


...

08번 폴드 완료 | 학습 2023개 | 검증 60개


09번 폴드 완료 | 학습 2205개 | 검증 60개


10번 폴드 완료 | 학습 2386개 | 검증 60개


11번 폴드 완료 | 학습 2568개 | 검증 60개


12번 폴드 완료 | 학습 2750개 | 검증 60개
전체 워크포워드 학습이 완료되었습니다.


In [6]:
# 폴드별 결과를 데이터프레임으로 변환합니다.
fold_results_df = pd.DataFrame(fold_results)

# 날짜 및 주요 평가 지표를 확인합니다.
display_columns = [
    "fold",
    "train_size",
    "valid_size",
    "train_end",
    "valid_start",
    "valid_end",
    "accuracy",
    "macro_f1",
    "balanced_accuracy",
    "direction_hit_rate",
    "always_up_accuracy",
    "three_class_majority_accuracy",
]

display(
    fold_results_df.loc[:, display_columns].round(4)
)

,fold,train_size,valid_size,train_end,valid_start,valid_end,accuracy,macro_f1,balanced_accuracy,direction_hit_rate,always_up_accuracy,three_class_majority_accuracy
0,1,750,60,20130401,20130409,20130704,0.3500,0.3369,0.3598,0.3243,0.2833,0.2833
1,2,932,60,20131224,20140106,20140401,0.3833,0.3343,0.3410,0.3077,0.2500,0.2500
2,3,1114,60,20140924,20141002,20141229,0.3500,0.3240,0.3214,0.2500,0.2667,0.4667
3,4,1295,60,20150619,20150629,20150921,0.4667,0.4619,0.4928,0.5122,0.2667,0.3167
4,5,1477,60,20160316,20160324,20160621,0.3667,0.3299,0.3483,0.1714,0.2333,0.4167
5,6,1659,60,20161208,20161216,20170315,0.5000,0.4122,0.4037,0.3333,0.3333,0.6000
6,7,1841,60,20170904,20170912,20171212,0.4000,0.2321,0.2618,0.0357,0.2500,0.5333
7,8,2023,60,20180607,20180618,20180910,0.2667,0.2335,0.2244,0.1613,0.2000,0.4833
8,9,2205,60,20190308,20190318,20190612,0.3833,0.3818,0.4050,0.4324,0.2667,0.3833
9,10,2386,60,20191128,20191206,20200305,0.4333,0.3707,0.4456,0.3953,0.3500,0.2833


In [7]:
# 12개 폴드의 검증 결과를 하나의 배열로 합칩니다.
oos_y_true = np.concatenate(all_y_true)
oos_y_pred = np.concatenate(all_y_pred)

# 기준선 예측 결과도 하나의 배열로 합칩니다.
oos_always_up_pred = np.concatenate(
    all_always_up_pred
)

oos_direction_majority_pred = np.concatenate(
    all_direction_majority_pred
)

oos_three_class_majority_pred = np.concatenate(
    all_three_class_majority_pred
)

# 전체 검증 결과를 요약합니다.
summary = pd.Series(
    {
        "사용 피처 수": len(COMBINATION_A),
        "워크포워드 폴드 수": len(fold_results_df),
        "전체 OOS 표본 수": len(oos_y_true),
        "평균 accuracy": fold_results_df[
            "accuracy"
        ].mean(),
        "accuracy 표준편차": fold_results_df[
            "accuracy"
        ].std(ddof=1),
        "평균 macro F1": fold_results_df[
            "macro_f1"
        ].mean(),
        "macro F1 표준편차": fold_results_df[
            "macro_f1"
        ].std(ddof=1),
        "전체 OOS accuracy": accuracy_score(
            oos_y_true,
            oos_y_pred,
        ),
        "전체 OOS macro F1": f1_score(
            oos_y_true,
            oos_y_pred,
            labels=CLASS_LABELS,
            average="macro",
            zero_division=0,
        ),
        "전체 OOS balanced accuracy": (
            balanced_accuracy_score(
                oos_y_true,
                oos_y_pred,
            )
        ),
        "전체 OOS 방향 적중률": hit_rate(
            oos_y_pred,
            oos_y_true,
        ),
        "항상 상승 accuracy": accuracy_score(
            oos_y_true,
            oos_always_up_pred,
        ),
        "3분류 다수 클래스 accuracy": accuracy_score(
            oos_y_true,
            oos_three_class_majority_pred,
        ),
        "항상 상승 방향 적중률": hit_rate(
            oos_always_up_pred,
            oos_y_true,
        ),
        "방향 다수 클래스 적중률": hit_rate(
            oos_direction_majority_pred,
            oos_y_true,
        ),
    },
    name="결과",
)

display(summary.to_frame().round(4))

,결과
사용 피처 수,5.0000
워크포워드 폴드 수,12.0000
전체 OOS 표본 수,720.0000
평균 accuracy,0.4097
accuracy 표준편차,0.0922
평균 macro F1,0.3461
macro F1 표준편차,0.0990
전체 OOS accuracy,0.4097
전체 OOS macro F1,0.3889
전체 OOS balanced accuracy,0.3942


In [8]:
# 혼동행렬의 행과 열에 사용할 클래스 이름을 준비합니다.
display_class_names = [
    "하락(-1)",
    "중립(0)",
    "상승(1)",
]

# 전체 OOS 예측 결과의 혼동행렬을 계산합니다.
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(
        oos_y_true,
        oos_y_pred,
        labels=CLASS_LABELS,
    ),
    index=[
        f"실제 {name}"
        for name in display_class_names
    ],
    columns=[
        f"예측 {name}"
        for name in display_class_names
    ],
)

display(confusion_matrix_df)

,예측 하락(-1),예측 중립(0),예측 상승(1)
실제 하락(-1),45,84,75
실제 중립(0),72,153,81
실제 상승(1),51,62,97


In [9]:
# 하락·중립·상승 클래스별 성능을 계산합니다.
classification_report_df = pd.DataFrame(
    classification_report(
        oos_y_true,
        oos_y_pred,
        labels=CLASS_LABELS,
        target_names=display_class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

display(classification_report_df.round(4))

,precision,recall,f1-score,support
하락(-1),0.2679,0.2206,0.2419,204.0000
중립(0),0.5117,0.5000,0.5058,306.0000
상승(1),0.3834,0.4619,0.4190,210.0000
accuracy,0.4097,0.4097,0.4097,0.4097
macro avg,0.3877,0.3942,0.3889,720.0000
weighted avg,0.4052,0.4097,0.4057,720.0000


# XGBoost 조합 A 실험 결과 해석

## 1. 혼동행렬

| 실제값 \ 예측값 | 하락(-1) | 중립(0) | 상승(1) |
| --- | ---: | ---: | ---: |
| 실제 하락(-1) | 13 | 108 | 83 |
| 실제 중립(0) | 25 | 182 | 99 |
| 실제 상승(1) | 24 | 87 | 99 |

혼동행렬은 행이 실제 정답이고 열이 모델 예측이다. 대각선은 맞힌 개수이고,
대각선 밖은 다른 클래스로 잘못 예측한 개수이다.

* 실제 하락 204개 중 13개를 하락으로 맞혔다.
* 실제 중립 306개 중 182개를 중립으로 맞혔다.
* 실제 상승 210개 중 99개를 상승으로 맞혔다.
* 전체 720개 중 `294개`를 올바르게 예측했다.
* 예측 분포는 하락 62개, 중립 377개, 상승 281개이다.

## 2. 클래스별 평가 지표

| 클래스 | Precision | Recall | F1-score | Support |
| --- | ---: | ---: | ---: | ---: |
| 하락(-1) | 0.2097 | 0.0637 | 0.0977 | 204 |
| 중립(0) | 0.4828 | 0.5948 | 0.5329 | 306 |
| 상승(1) | 0.3523 | 0.4714 | 0.4033 | 210 |

Precision은 해당 클래스로 예측한 것 중 맞힌 비율이고, Recall은 실제 해당 클래스를
찾아낸 비율이다. F1-score는 Precision과 Recall의 조화평균이므로 어느 한쪽의 낮은
값을 숨기지 않는다. Support는 전체 OOS 검증 데이터의 실제 클래스별 개수이다.

## 3. 전체 성능

| 평가 지표 | 결과 |
| --- | ---: |
| Accuracy | 0.4083 |
| Macro F1-score | 0.3446 |
| Balanced Accuracy | 0.3766 |

Accuracy는 전체 정답 비율이다. Macro F1은 하락·중립·상승을 같은 비중으로 평가하고,
Balanced Accuracy는 세 클래스 Recall의 평균이다. 중립 표본이 가장 많으므로 Accuracy만
보면 소수 클래스인 하락을 놓치는 문제를 과소평가할 수 있다.

## 4. Logistic Regression 및 기준선 비교

Logistic Regression 조합 A의 실측값은 Accuracy 0.3958, Macro F1 0.2971,
하락 Recall 0.0000이다. 세 트리 모델 중 Accuracy와 Balanced Accuracy가 가장 높고 Logistic보다 Accuracy가 1.25%p 높다. 그러나 하락 Recall은 6.37%에 그쳐 중립·상승 예측 쏠림이 남았고, 최빈 클래스 기준선 Accuracy 0.4250은 넘지 못했다.

## 5. 결론

앞선 트리의 오차를 다음 트리가 보완하는 부스팅 모델이며, 프로젝트 라벨 -1·0·1을 내부에서 0·1·2로 변환한다. 트리 기반이라 StandardScaler를 사용하지 않는다. 이번 결과는 기본 하이퍼파라미터 기준선이며 최적화 결과가 아니다.
같은 조합 A와 같은 워크포워드 구간에서 모델 구조만 바꿨으므로, 비선형 트리 모델은
Logistic이 전혀 찾지 못한 하락을 일부 구분한다는 점은 확인됐다. 다만 어느 모델도
최빈 클래스 기준선 Accuracy를 넘지 못했으므로 실전 매매 신호로 채택할 근거는 없다.
